# 🖼️ Laboratorio 4: Redes Neuronales Convolucionales (CNN)
## Caso de Estudio: Clasificación de Imágenes — de los Filtros Sobel a las *Feature Maps*

**Pontificia Universidad Javeriana Cali — Curso de Deep Learning**  
**Tutor:** Jan Polanco Velasco  
**Semestre:** 2026-2

---

### 🎯 Objetivo del Laboratorio

En el **Laboratorio 1** pronosticamos series temporales con una **MLP autorregresiva** (vector plano); en el **Lab 2** exploramos la **memoria secuencial** (RNN/LSTM/GRU); en el **Lab 3** convertimos audio en imágenes tiempo-frecuencia y las clasificamos con una CNN. En este laboratorio **abrimos la caja negra de la convolución**: implementamos la operación 2D a mano, estudiamos su **geometría** (padding, stride, volúmenes multicanal), el **pooling** y la **jerarquía visual**, y entrenamos una **CNN canónica** sobre **Fashion-MNIST** (imágenes reales de prendas de vestir, 28×28).

El material sigue los **4 actos del deck de la Sesión 4**: (1) de la imagen al tensor y el colapso de la MLP, (2) la operación de convolución 2D y los kernels clásicos vs. aprendidos, (3) la geometría de la convolución (padding/stride/3D) y (4) pooling, jerarquía y arquitectura canónica en Keras.

### 📋 Contenido

1. Marco teórico: píxeles, tensores y convolución
2. Configuración inicial
3. Acto 1 — De la imagen al tensor: colapso de la MLP y explosión de parámetros
4. Acto 2 — Convolución 2D manual: kernels Sobel, Laplaciano y filtros aprendidos
5. Acto 3 — Geometría: padding, stride y convolución sobre volúmenes 3D
6. Acto 4 — Pooling, jerarquía visual y arquitectura canónica
7. Entrenamiento de la CNN sobre Fashion-MNIST
8. Evaluación y mapas de activación (*feature maps*)
9. Conclusiones

> 💡 **Consejo:** ejecuta las celdas en orden. Las gráficas de Plotly son interactivas (*zoom*, *hover* y leyenda). El notebook es *clean* (sin salidas versionadas): ejecútalo para reproducir los resultados.

---
## 1. 📚 Marco Teórico: de los Píxeles a la Convolución

### 1.1 La imagen como tensor

Una imagen en escala de grises es una matriz $H \times W$; una imagen a color añade **canales**:

$$\mathbf{X} \in \mathbb{R}^{H \times W \times C}, \qquad C = 3 \text{ (RGB)}$$

En Keras, un lote de imágenes es un tensor **4D**: `[batch, height, width, channels]`. En este laboratorio clasificaremos imágenes de $28 \times 28 \times 1$.

### 1.2 ¿Por qué no una MLP? El colapso del *Flatten*

- **Pérdida de topología:** al vectorizar $2D \to 1D$, dos píxeles vecinos verticalmente quedan separados en el vector; la noción de *vecindad* se destruye.
- **Explosión de parámetros:** una imagen $200 \times 200 \times 3$ con $1\,000$ neuronas ocultas exige $120\,000 \times 1\,000 = 120$ millones de pesos.
- **Cero invarianza a traslación:** desplazar el objeto unos píxeles produce un vector totalmente distinto y desconectado.

### 1.3 La convolución 2D

Una capa `Conv2D` desliza $K$ filtros $f \times f$ y calcula un producto punto local. En *Deep Learning* la "convolución" es en realidad **correlación cruzada** (el kernel no se voltea):

$$S(i,j) = (X * K)(i,j) + b = \sum_m \sum_n X_{i+m,\,j+n}\,K_{m,n} + b$$

Dos principios la distinguen de una capa densa:

1. **Conexiones locales:** cada neurona observa solo un parche diminuto.
2. **Pesos compartidos:** el mismo filtro barre **toda** la imagen.

### 1.4 Geometría y pooling

La dimensión de salida de una capa convolucional o de pooling sigue la **ecuación universal**:

$$O = \left\lfloor \frac{N + 2p - f}{s} \right\rfloor + 1$$

donde $N$ es el tamaño de entrada, $p$ el *padding*, $f$ el tamaño del kernel y $s$ el *stride*. Si $C_{in}$ es el número de canales de entrada y $K$ el número de filtros, los parámetros de la capa son $(f \cdot f \cdot C_{in} + 1)\cdot K$.

| Operación | Efecto | Parámetros |
|---|---|---|
| `Conv2D` | Detecta motivos locales con pesos compartidos | $(f^2 C_{in} + 1)K$ |
| `MaxPooling2D` | Submuestrea e introduce invarianza local | 0 |
| `GlobalAveragePooling2D` | Un valor por mapa → vector | 0 |
| `Flatten` | Vectoriza todas las posiciones | 0 (pero dispara la densa) |

---
## 2. ⚙️ Configuración Inicial

In [ ]:
import time
import warnings

import numpy as np
import plotly.graph_objects as go
import tensorflow as tf
from numpy.lib.stride_tricks import sliding_window_view
from plotly.subplots import make_subplots
from scipy.ndimage import correlate
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    Activation,
    BatchNormalization,
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    GlobalAveragePooling2D,
    Input,
    MaxPooling2D,
)
from tensorflow.keras.models import Model, Sequential

warnings.filterwarnings("ignore")

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version:      {tf.keras.__version__}")
print(f"GPU disponible:     {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## 3. 🧱 Acto 1 — De la Imagen al Tensor

Trabajaremos con **Fashion-MNIST**: 70 000 imágenes reales de prendas de vestir en escala de grises ($28 \times 28$), organizadas en 10 clases. Primero cargamos el dataset y observamos una muestra de cada clase; después construimos una **escena sintética** con bordes controlados para estudiar los filtros.

In [ ]:
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = tf.keras.datasets.fashion_mnist.load_data()

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

print(f"Train: {x_train_raw.shape} | Test: {x_test_raw.shape} | Clases: {len(CLASS_NAMES)}")
print(f"Etiquetas train: {np.bincount(y_train_raw)}")

fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=[CLASS_NAMES[c] for c in range(10)],
    horizontal_spacing=0.03, vertical_spacing=0.16,
)
for c in range(10):
    idx = int(np.where(y_train_raw == c)[0][0])
    fig.add_trace(
        go.Heatmap(z=x_train_raw[idx], colorscale="gray", showscale=False),
        row=c // 5 + 1, col=c % 5 + 1,
    )
fig.update_layout(height=480, title="Fashion-MNIST: una imagen real por clase (28×28)")
fig.show()

### 3.1 Una escena sintética para estudiar bordes

Para visualizar con claridad el efecto de los filtros, generamos una escena de $128 \times 128$ con formas geométricas bien definidas (un rectángulo brillante, un círculo oscuro, una diagonal y un fondo con gradiente). Al mantener el control sobre sus bordes sabemos *qué esperar* de Sobel y del Laplaciano.

In [ ]:
def make_scene(size=128):
    """Escena sintética en escala de grises con bordes bien definidos."""
    yy, xx = np.mgrid[0:size, 0:size].astype(float)
    img = 40 + 60 * (xx / size)                  # gradiente de fondo
    img[20:70, 20:60] = 220                      # rectángulo brillante
    circle = (yy - 88) ** 2 + (xx - 92) ** 2 <= 22 ** 2
    img[circle] = 30                             # círculo oscuro
    for i in range(size):                        # diagonal
        img[max(0, i - 1):i + 2, max(0, i - 1):i + 2] = 255
    rng = np.random.default_rng(0)
    img = img + rng.normal(0, 2.5, img.shape)    # ruido leve
    return np.clip(img, 0, 255).astype("float32")


image = make_scene()
print(f"Tensor de la imagen: {image.shape}  (alto × ancho)")
print(f"Rango de intensidad: [{image.min():.1f}, {image.max():.1f}]")

fig = go.Figure(go.Heatmap(z=image, colorscale="gray", zmin=0, zmax=255, showscale=False))
fig.update_layout(
    title="Escena sintética (escala de grises)",
    height=430, xaxis_title="Ancho (W)", yaxis_title="Alto (H)",
    yaxis_autorange="reversed",
)
fig.show()

### 3.2 El costo de aplanar (*Flatten*)

Aplanar una imagen para una capa densa no solo destruye la vecindad espacial: también hace explotar el número de parámetros. Comparemos una MLP ingenua con una **única** capa convolucional sobre una imagen de $200 \times 200 \times 3$.

In [ ]:
H = W = 200
C = 3
HIDDEN = 1000

mlp_params = (H * W * C) * HIDDEN + HIDDEN
filters, f = 32, 3
conv_params = (f * f * C + 1) * filters

print(f"MLP: {H}×{W}×{C} = {H * W * C:,} entradas × {HIDDEN} neuronas → {mlp_params:,} parámetros")
print(f"Conv2D({filters}, {f}×{f}) sobre la misma imagen      → {conv_params:,} parámetros")
print(f"Reducción: {mlp_params / conv_params:,.0f}× menos pesos")

fig = go.Figure(
    go.Bar(
        x=["MLP (Flatten)", f"Conv2D ({filters} filtros)"],
        y=[mlp_params, conv_params],
        marker_color=["#ef4444", "#4f46e5"],
        text=[f"{mlp_params:,}", f"{conv_params:,}"],
        textposition="outside",
    )
)
fig.update_layout(
    title="Explosión de parámetros: MLP densa vs. capa convolucional",
    yaxis_title="Parámetros (escala log)", yaxis_type="log", height=400,
)
fig.show()

---
## 4. 🔎 Acto 2 — La Operación de Convolución 2D

### 4.1 Implementación a mano

Escribimos la convolución (correlación cruzada) con NumPy: deslizamos el kernel sobre la imagen, multiplicamos elemento a elemento y sumamos. Usamos `sliding_window_view` para vectorizar el barrido y una versión con bucles explícitos como referencia didáctica.

In [ ]:
def conv2d_valid(image, kernel):
    """Correlación cruzada 2D 'valid' con bucles explícitos (versión didáctica)."""
    image, kernel = np.asarray(image, float), np.asarray(kernel, float)
    H, W = image.shape
    f = kernel.shape[0]
    out = np.zeros((H - f + 1, W - f + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(image[i:i + f, j:j + f] * kernel)
    return out


def conv2d_vec(image, kernel):
    """Misma operación, vectorizada con ventanas deslizantes (rápida)."""
    image, kernel = np.asarray(image, float), np.asarray(kernel, float)
    windows = sliding_window_view(image, kernel.shape)
    return np.einsum("ijkl,kl->ij", windows, kernel)

### 4.2 El ejemplo del componente `ConvolutionSimulator`

Reproducimos en Python la misma matriz $5 \times 5$ y los mismos kernels del componente interactivo del deck: **Sobel Vertical**, **Laplaciano** e **Identidad**. El resultado es exactamente la salida $3 \times 3$ que se muestra en las diapositivas.

In [ ]:
X_deck = np.array([
    [10, 10, 10, 200, 200],
    [10, 10, 10, 200, 200],
    [10, 10, 10, 200, 200],
    [10, 10, 10, 200, 200],
    [10, 10, 10, 200, 200],
], dtype=float)

kernels = {
    "Sobel Vertical": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float),
    "Laplaciano": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], float),
    "Identidad": np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], float),
}

for name, K in kernels.items():
    out = conv2d_valid(X_deck, K)
    print(f"{name}  →\n{out.astype(int)}\n")

In [ ]:
fig = make_subplots(rows=1, cols=4, subplot_titles=["Entrada 5×5", "Sobel Vertical", "Laplaciano", "Identidad"])
fig.add_trace(go.Heatmap(z=X_deck, colorscale="gray", showscale=False, text=X_deck.astype(int), texttemplate="%{text}"), row=1, col=1)
for col, name in enumerate(["Sobel Vertical", "Laplaciano", "Identidad"], start=2):
    out = conv2d_valid(X_deck, kernels[name])
    fig.add_trace(
        go.Heatmap(z=out, colorscale="RdBu", zmid=0, showscale=False, text=out.astype(int), texttemplate="%{text}"),
        row=1, col=col,
    )
fig.update_layout(height=360, title="Convolución 2D a mano: el ejemplo del deck")
fig.show()

### 4.3 Validación contra `scipy` y costo computacional

Comprobamos que nuestra implementación coincide con `scipy.ndimage.correlate` y estimamos el número de multiplicaciones-acumulaciones (MACs) de la operación.

In [ ]:
K = kernels["Sobel Vertical"]
mi = conv2d_vec(image, K)

ref_full = correlate(image, K, mode="constant", cval=0.0)
p = K.shape[0] // 2
ref = ref_full[p:p + mi.shape[0], p:p + mi.shape[1]]

print("¿Coinciden nuestra convolución y scipy.ndimage.correlate?", np.allclose(mi, ref))

macs = mi.shape[0] * mi.shape[1] * K.size
print(f"MACs por filtro sobre la escena {image.shape}: {macs:,}")
print(f"MACs para 32 filtros: {macs * 32:,} — trivial para una GPU.")

### 4.4 Equivarianza de la convolución e invarianza del pooling

La convolución es **equivariante**: si el objeto se traslada, su mapa de características se traslada igual. El **pooling global** convierte esa equivarianza en **invarianza**: al resumir cada mapa en un promedio, la representación deja de depender de la posición. Una MLP de vector plano, en cambio, ve un vector completamente distinto.

In [ ]:
# Un objeto sobre fondo uniforme, desplazado 4 píxeles
obj = np.zeros((64, 64), "float32")
obj[24:40, 24:40] = 200.0
obj_shift = np.roll(obj, 4, axis=1)


def descriptor(img):
    """Convolución con un banco de filtros + Global Average Pooling."""
    bank = [kernels["Sobel Vertical"], kernels["Sobel Vertical"].T, kernels["Laplaciano"]]
    return np.array([np.abs(conv2d_vec(img, K)).mean() for K in bank])


d_flat = np.linalg.norm(obj.ravel() - obj_shift.ravel()) / np.linalg.norm(obj.ravel())
desc, desc_shift = descriptor(obj), descriptor(obj_shift)
d_cnn = np.linalg.norm(desc - desc_shift) / (np.linalg.norm(desc) + 1e-9)

print(f"Distancia relativa (vector plano, MLP):  {d_flat:.4f}")
print(f"Distancia relativa (conv + GAP, CNN):    {d_cnn:.4f}")
print("→ Aplanar se rompe ante la traslación; convolución + pooling global es invariante.")

### 4.5 Filtros clásicos sobre una imagen real

Aplicamos Sobel (X e Y), la magnitud del gradiente y el Laplaciano a la escena sintética y a una imagen real de Fashion-MNIST. Los filtros clásicos son **fijos**; en una CNN sus valores serán **parámetros entrenables**.

In [ ]:
Kx = kernels["Sobel Vertical"]
Ky = Kx.T                                  # Sobel horizontal (detecta bordes horizontales)
Klap = kernels["Laplaciano"]

sobel_x = conv2d_vec(image, Kx)
sobel_y = conv2d_vec(image, Ky)
magnitude = np.hypot(sobel_x, sobel_y)
laplacian = conv2d_vec(image, Klap)

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=["Original", "|Sobel X| (bordes verticales)", "|Sobel Y| (bordes horizontales)", "Magnitud del gradiente"],
)
for col, z in enumerate([image, np.abs(sobel_x), np.abs(sobel_y), magnitude], start=1):
    fig.add_trace(go.Heatmap(z=z, colorscale="gray" if col == 1 else "Inferno", showscale=False), row=1, col=col)
fig.update_layout(height=380, title="Filtros clásicos sobre la escena sintética")
fig.show()

In [ ]:
real_idx = int(np.where(y_train_raw == 9)[0][0])   # una bota: bordes nítidos
real = x_train_raw[real_idx].astype(float)

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[f"Real: {CLASS_NAMES[y_train_raw[real_idx]]}", "|Sobel X|", "|Sobel Y|", "Laplaciano"],
)
fig.add_trace(go.Heatmap(z=real, colorscale="gray", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=np.abs(conv2d_vec(real, Kx)), colorscale="Inferno", showscale=False), row=1, col=2)
fig.add_trace(go.Heatmap(z=np.abs(conv2d_vec(real, Ky)), colorscale="Inferno", showscale=False), row=1, col=3)
fig.add_trace(go.Heatmap(z=np.abs(conv2d_vec(real, Klap)), colorscale="Inferno", showscale=False), row=1, col=4)
fig.update_layout(height=360, title="Filtros clásicos sobre una imagen real (28×28)")
fig.show()

---
## 5. 📐 Acto 3 — Geometría de la Convolución

### 5.1 La ecuación dimensional universal

$$O = \left\lfloor \frac{N + 2p - f}{s} \right\rfloor + 1$$

- `padding="valid"` → $p = 0$: la salida **encoge** y los bordes participan menos.
- `padding="same"` → la salida conserva $H \times W$ (para $s = 1$).
- `strides > 1` → **submuestreo** aprendible.

Verifiquemos la fórmula contra la forma real que reporta Keras.

In [ ]:
def out_size(n, f, p, s):
    return (n + 2 * p - f) // s + 1


casos = [
    ("valid, f=3, s=1", 28, 3, 0, 1),
    ("same,  f=3, s=1", 28, 3, 1, 1),
    ("same,  f=3, s=2", 28, 3, 1, 2),
    ("valid, f=5, s=1", 28, 5, 0, 1),
]

print(f"{'Configuración':<18}{'O fórmula':>10}{'O Keras':>10}")
for name, n, f, p, s in casos:
    m = Sequential([Input(shape=(n, n, 1)), Conv2D(4, f, padding="same" if p else "valid", strides=s)])
    print(f"{name:<18}{out_size(n, f, p, s):>10}{m.output_shape[1]:>10}")

In [ ]:
padded = np.pad(image, 1, mode="constant")
out_same = conv2d_vec(padded, np.ones((3, 3)))
print(f"p=1 → {image.shape} → padding {padded.shape} → salida {out_same.shape} (equivale a 'same')")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Original 128×128", "Con padding de ceros 130×130"])
fig.add_trace(go.Heatmap(z=image, colorscale="gray", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=padded, colorscale="gray", showscale=False), row=1, col=2)
fig.update_layout(height=380, title="Padding 'same': preservar la resolución espacial")
fig.show()

In [ ]:
strided = Sequential([Input(shape=(28, 28, 1)), Conv2D(1, 3, strides=2, padding="same", use_bias=False)]).output_shape
pooled = Sequential([Input(shape=(28, 28, 1)), MaxPooling2D(2)]).output_shape
print(f"Conv2D(stride=2, same) → {strided[1:3]}   |   MaxPooling2D(2) → {pooled[1:3]}")
print("Ambas reducen 28→14, pero la convolución con stride es APRENDIBLE y el pooling NO.")

### 5.2 Convolución sobre volúmenes multicanal (3D)

Con una entrada RGB, cada filtro es un volumen $f \times f \times C_{in}$ y produce **un solo** mapa 2D. Con $K$ filtros, la salida tiene $K$ canales:

$$(H \times W \times C_{in}) \xrightarrow{\;K \text{ filtros } f\times f\;} (H' \times W' \times K)$$

La profundidad de la salida **no depende** de los canales de entrada, sino de cuántos filtros configuremos. Los parámetros de la capa son $(f \cdot f \cdot C_{in} + 1)\cdot K$.

In [ ]:
tf.keras.utils.set_random_seed(42)

rgb = np.zeros((32, 32, 3), "float32")
rgb[:, :, 0] = 0.80                      # fondo rojizo tenue
rgb[8:24, 8:16] = [0.0, 0.9, 0.2]        # rectángulo verde
rgb[10:30, 22:30] = [0.1, 0.2, 0.95]     # rectángulo azul
rgb[4:10, 20:28] = [0.95, 0.9, 0.1]      # rectángulo amarillo

conv_rgb = Conv2D(16, 3, padding="same", activation="relu", name="conv_rgb")
out = conv_rgb(rgb[None, ...])
print(f"Entrada {rgb[None, ...].shape} → salida {out.shape}")
print(f"Parámetros: (3·3·3 + 1)·16 = {(3 * 3 * 3 + 1) * 16}")

fig_img = go.Figure(go.Image(z=(rgb * 255).astype("uint8")))
fig_img.update_layout(title="Imagen RGB de entrada (32×32×3)", height=360)
fig_img.show()

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=[f"Filtro {i}" for i in range(8)],
    horizontal_spacing=0.04, vertical_spacing=0.14,
)
for i in range(8):
    fig.add_trace(
        go.Heatmap(z=out[0, :, :, i].numpy(), colorscale="Viridis", showscale=False),
        row=i // 4 + 1, col=i % 4 + 1,
    )
fig.update_layout(height=500, title="Convolución 3D: 8 de los 16 mapas de características")
fig.show()

---
## 6. 🧩 Acto 4 — Pooling, Jerarquía y Arquitectura CNN

### 6.1 Pooling a mano

El *pooling* resume la **presencia** de una característica en una región: reduce ancho y alto sin tocar los canales y **no añade parámetros**. Su dimensión de salida también sigue $O = \lfloor (N - f)/s \rfloor + 1$.

In [ ]:
def max_pool2d(x, f=2, s=2):
    H, W = x.shape
    oh, ow = (H - f) // s + 1, (W - f) // s + 1
    return np.array([[x[i * s:i * s + f, j * s:j * s + f].max() for j in range(ow)] for i in range(oh)])


def avg_pool2d(x, f=2, s=2):
    H, W = x.shape
    oh, ow = (H - f) // s + 1, (W - f) // s + 1
    return np.array([[x[i * s:i * s + f, j * s:j * s + f].mean() for j in range(ow)] for i in range(oh)])


M = np.array([[5, 6, 6, 5], [9, 1, 2, 8], [4, 7, 1, 8], [3, 2, 2, 7]], dtype=float)
M_max, M_avg = max_pool2d(M), avg_pool2d(M)
print("Max Pooling:\n", M_max, "\nAverage Pooling:\n", np.round(M_avg, 2))

fig = make_subplots(rows=1, cols=3, subplot_titles=["Entrada (4×4)", "Max Pooling (2×2)", "Average Pooling (2×2)"])
fig.add_trace(go.Heatmap(z=M, colorscale="Blues", text=M.astype(int), texttemplate="%{text}", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=M_max, colorscale="Reds", text=M_max.astype(int), texttemplate="%{text}", showscale=False), row=1, col=2)
fig.add_trace(go.Heatmap(z=M_avg, colorscale="Greens", text=np.round(M_avg, 2), texttemplate="%{text}", showscale=False), row=1, col=3)
fig.update_layout(height=360, title="Max vs. Average Pooling (f = 2, s = 2)")
fig.show()

In [ ]:
# El mismo "punto caliente" en dos posiciones DENTRO de la misma ventana 2×2
a = np.zeros((2, 2)); a[0, 0] = 1.0
b = np.zeros((2, 2)); b[1, 1] = 1.0

print("MaxPooling de dos posiciones dentro de la misma ventana:")
print("  a →", max_pool2d(a), "| b →", max_pool2d(b))
print("  ¿idénticos?", bool(np.allclose(max_pool2d(a), max_pool2d(b))))
print("→ El pooling es invariante a traslaciones pequeñas dentro de su ventana.")

### 6.2 Arquitectura canónica en Keras

La receta estándar: un **extractor de características** (`Conv2D + BatchNorm + ReLU + MaxPooling` apilados, donde $H, W \downarrow$ y los canales $\uparrow$) y una **cabeza clasificadora** (`GlobalAveragePooling2D → Dropout → Dense(softmax)`).

| Bloque | Capa | Salida |
|---|---|---|
| Entrada | `Input(28, 28, 1)` | $(28, 28, 1)$ |
| 1 | `Conv2D(32, 3, same)` + `BN` + `ReLU` + `MaxPool(2)` | $(14, 14, 32)$ |
| 2 | `Conv2D(64, 3, same)` + `BN` + `ReLU` + `MaxPool(2)` | $(7, 7, 64)$ |
| 3 | `Conv2D(64, 3, same)` + `BN` + `ReLU` | $(7, 7, 64)$ |
| Cabeza | `GAP` → `Dropout(0.3)` → `Dense(10, softmax)` | $(10,)$ |

Usamos `BatchNormalization` para estabilizar y acelerar la optimización (lo aprendido en la Sesión 3). `GlobalAveragePooling2D` evita la capa densa gigante de `Flatten`: cada mapa vota por la **presencia** de su patrón, aportando invarianza espacial con muy pocos parámetros.

In [ ]:
def build_cnn():
    tf.keras.utils.set_random_seed(42)
    model = Sequential(
        [
            Input(shape=(28, 28, 1), name="imagen"),
            Conv2D(32, 3, padding="same", use_bias=False, name="conv1"),
            BatchNormalization(name="bn1"),
            Activation("relu", name="relu1"),
            MaxPooling2D(2, name="pool1"),
            Conv2D(64, 3, padding="same", use_bias=False, name="conv2"),
            BatchNormalization(name="bn2"),
            Activation("relu", name="relu2"),
            MaxPooling2D(2, name="pool2"),
            Conv2D(64, 3, padding="same", use_bias=False, name="conv3"),
            BatchNormalization(name="bn3"),
            Activation("relu", name="relu3"),
            GlobalAveragePooling2D(name="gap"),
            Dropout(0.3, name="dropout"),
            Dense(10, activation="softmax", name="clasificador"),
        ],
        name="CNN_FashionMNIST",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_cnn()
model.summary()
print(f"\nParámetros totales: {model.count_params():,}")

In [ ]:
# Normalizamos a [0, 1] y añadimos el canal (N, 28, 28, 1)
x_train = (x_train_raw.astype("float32") / 255.0)[..., None]
x_test = (x_test_raw.astype("float32") / 255.0)[..., None]

print(f"x_train {x_train.shape} | x_test {x_test.shape}")
print(f"Rango de píxeles: [{x_train.min():.1f}, {x_train.max():.1f}]")
print(f"Balance de clases (train): {np.bincount(y_train_raw)}")

### 6.3 Entrenamiento

Entrenamos con `Adam`, la pérdida `sparse_categorical_crossentropy` y `EarlyStopping(patience=3, restore_best_weights=True)`. Reservamos el 10 % del *train* como validación; el conjunto de **test** queda intacto hasta la evaluación final.

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1)

start = time.perf_counter()
history = model.fit(
    x_train, y_train_raw,
    validation_split=0.1,
    epochs=12,
    batch_size=128,
    callbacks=[early_stop],
    verbose=2,
)
elapsed = time.perf_counter() - start

print(f"\nEntrenado en {elapsed:.1f} s · {len(history.history['loss'])} épocas")
print(f"Mejor val_accuracy: {max(history.history['val_accuracy']):.4f}")

In [ ]:
epochs = list(range(1, len(history.history["loss"]) + 1))
fig = make_subplots(rows=1, cols=2, subplot_titles=["Función de costo (cross-entropy)", "Accuracy"])
fig.add_trace(go.Scatter(x=epochs, y=history.history["loss"], mode="lines", name="train", line=dict(color="#4f46e5", width=2.2)), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs, y=history.history["val_loss"], mode="lines", name="validación", line=dict(color="#ef4444", width=2.2, dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs, y=history.history["accuracy"], mode="lines", name="train", line=dict(color="#4f46e5", width=2.2), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs, y=history.history["val_accuracy"], mode="lines", name="validación", line=dict(color="#ef4444", width=2.2, dash="dash"), showlegend=False), row=1, col=2)
fig.update_layout(height=380, title="Curvas de aprendizaje — diagnóstico del entrenamiento")
fig.update_xaxes(title_text="Época")
fig.show()

---
## 7. 🎯 Evaluación y Mapas de Activación

### 7.1 Desempeño en el conjunto de prueba

El *test* (10 000 imágenes) nunca se usó para entrenar ni para *early stopping*: mide generalización real. Reportamos *accuracy*, `classification_report` y la **matriz de confusión**.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test_raw, verbose=0)
print(f"Test loss: {test_loss:.4f} · Test accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
print(classification_report(y_test_raw, y_pred, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_test_raw, y_pred)
fig = go.Figure(data=go.Heatmap(z=cm, x=CLASS_NAMES, y=CLASS_NAMES, colorscale="Blues", showscale=True))
for i in range(10):
    for j in range(10):
        fig.add_annotation(
            x=CLASS_NAMES[j], y=CLASS_NAMES[i], text=str(cm[i, j]), showarrow=False,
            font=dict(color="white" if cm[i, j] > cm.max() / 2 else "black", size=10),
        )
fig.update_layout(
    title=f"Matriz de confusión (test, acc={test_acc:.2f})",
    xaxis_title="Predicción", yaxis_title="Real", height=620,
)
fig.show()

### 7.2 Qué ven los filtros: la jerarquía visual

Abrimos la caja negra extrayendo los **mapas de activación** de `conv1` y `conv2`. Observa cómo las capas iniciales responden a **bordes** (mapas de $28 \times 28$) y las intermedias combinan esos bordes en **partes** de mayor nivel (mapas de $14 \times 14$).

In [ ]:
probe_idx = int(np.where(y_test_raw == 9)[0][0])   # una bota
activation_model = Model(
    inputs=model.inputs,
    outputs=[model.get_layer("conv1").output, model.get_layer("conv2").output],
)
a1, a2 = activation_model.predict(x_test[probe_idx:probe_idx + 1], verbose=0)
print(f"conv1: {a1.shape} (28×28) | conv2: {a2.shape} (14×14)")

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=[f"conv1 · filtro {i}" for i in range(4)] + [f"conv2 · filtro {i}" for i in range(4)],
    vertical_spacing=0.14,
)
for i in range(4):
    fig.add_trace(go.Heatmap(z=a1[0, :, :, i], colorscale="Viridis", showscale=False), row=1, col=i + 1)
    fig.add_trace(go.Heatmap(z=a2[0, :, :, i], colorscale="Viridis", showscale=False), row=2, col=i + 1)
fig.update_layout(height=520, title="Jerarquía: conv1 (bordes, 28×28) vs. conv2 (partes, 14×14)")
fig.show()

In [ ]:
w = model.get_layer("conv1").get_weights()[0]
print("Forma de los filtros conv1:", w.shape, "(alto, ancho, canales_in, filtros)")

fig = make_subplots(
    rows=2, cols=8, subplot_titles=[f"{i}" for i in range(16)],
    horizontal_spacing=0.04, vertical_spacing=0.18,
)
for i in range(16):
    fig.add_trace(
        go.Heatmap(z=w[:, :, 0, i], colorscale="RdBu", zmid=0, showscale=False),
        row=i // 8 + 1, col=i % 8 + 1,
    )
fig.update_layout(height=360, title="16 de los 32 filtros 3×3 aprendidos por conv1")
fig.show()

### 7.3 ¿Vale la pena la convolución? MLP vs. CNN

Comparamos la CNN (~57 k parámetros) con una **MLP de vector plano** más grande (~200 k parámetros) entrenada sobre la misma muestra de 15 000 imágenes. La CNN logra un desempeño comparable (o superior) con **muchos menos parámetros**: los pesos compartidos y la localidad actúan como una regularización estructural. Su ventaja se dispara a medida que crece la resolución de la imagen.

In [ ]:
BENCH_N = 15000
x_bench, y_bench = x_train[:BENCH_N], y_train_raw[:BENCH_N]

tf.keras.utils.set_random_seed(42)
mlp = Sequential(
    [
        Input(shape=(28, 28, 1)),
        Flatten(name="flatten"),
        Dense(256, activation="relu", name="oculta"),
        Dropout(0.3),
        Dense(10, activation="softmax", name="salida"),
    ],
    name="MLP_flatten",
)
mlp.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
mlp.fit(x_bench, y_bench, validation_split=0.1, epochs=6, batch_size=128, verbose=0)
mlp_acc = mlp.evaluate(x_test, y_test_raw, verbose=0)[1]

print(f"{'Modelo':<22}{'Parámetros':>12}{'Test acc':>12}")
print(f"{'MLP (Flatten)':<22}{mlp.count_params():>12,}{mlp_acc:>12.4f}")
print(f"{'CNN (convolución)':<22}{model.count_params():>12,}{test_acc:>12.4f}")
print(f"\nLa CNN usa {mlp.count_params() / model.count_params():.1f}× menos parámetros que la MLP.")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Test accuracy", "Parámetros"])
fig.add_trace(
    go.Bar(
        x=["MLP (Flatten)", "CNN"], y=[mlp_acc, test_acc],
        marker_color=["#ef4444", "#4f46e5"], text=[f"{mlp_acc:.3f}", f"{test_acc:.3f}"],
        textposition="outside", showlegend=False,
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(
        x=["MLP (Flatten)", "CNN"], y=[mlp.count_params(), model.count_params()],
        marker_color=["#ef4444", "#4f46e5"], text=[f"{mlp.count_params():,}", f"{model.count_params():,}"],
        textposition="outside", showlegend=False,
    ),
    row=1, col=2,
)
fig.update_yaxes(range=[0, 1], row=1, col=1)
fig.update_layout(height=400, title="MLP vs. CNN: desempeño y eficiencia de parámetros")
fig.show()

---
## 8. 🎓 Conclusiones

1. **La imagen es un tensor con topología:** `[batch, alto, ancho, canales]`. Aplanarla (`Flatten`) destruye la vecindad espacial y dispara los parámetros (120 M frente a ~900 en una capa convolucional del mismo ejemplo).
2. **La convolución es un producto punto local con pesos compartidos:** la implementamos a mano, la validamos contra `scipy.ndimage.correlate` y comprobamos que detecta bordes (Sobel) y cambios omnidireccionales (Laplaciano). En una CNN esos kernels **se aprenden**.
3. **La geometría se predice con una sola fórmula:** $O = \lfloor (N + 2p - f)/s \rfloor + 1$. Verificamos `valid`/`same` y `stride`, y vimos que el *stride* es un submuestreo **aprendible** frente al pooling, que no tiene parámetros.
4. **Volúmenes 3D:** con entrada RGB cada filtro es $f \times f \times C_{in}$ y produce un mapa; la profundidad de salida la fija el número de filtros, no los canales de entrada. Parámetros: $(f^2 C_{in} + 1)K$.
5. **El pooling aporta invarianza local:** MaxPooling conserva el valor dominante y es insensible a traslaciones pequeñas dentro de su ventana; `GlobalAveragePooling2D` resume cada mapa en un solo número y evita la capa densa gigante de `Flatten`.
6. **La jerarquía visual emerge sola:** `conv1` responde a bordes (28×28) y `conv2` a partes más complejas (14×14). La CNN aprovecha la estructura espacial y generaliza mejor que una MLP de vector plano.

### 🚀 Próximos pasos

- **Data augmentation** (`RandomFlip`, `RandomRotation`, `RandomZoom`) para mejorar la generalización.
- **Transfer learning**: reutilizar una CNN preentrenada (VGG, ResNet) y ajustar la cabeza.
- **Arquitecturas modernas:** bloques residuales (`ResNet`), convoluciones separables (`MobileNet`) y mecanismos de atención (`Vision Transformer`).
- **Interpretabilidad:** mapas de saliencia (*Grad-CAM*) para localizar qué mira la red al clasificar.

---

**📚 Referencias:**

[1] LeCun, Y. et al. (1998). *Gradient-Based Learning Applied to Document Recognition*. Proceedings of the IEEE — convoluciones, pooling y LeNet.  
[2] Krizhevsky, A., Sutskever, I. & Hinton, G. (2012). *ImageNet Classification with Deep Convolutional Neural Networks*. NeurIPS — la revolución de las CNN.  
[3] Xiao, H., Rasul, K. & Vollgraf, R. (2017). *Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms*. arXiv:1708.07747.  
[4] Chollet, F. (2021). *Deep Learning with Python*. 2nd ed. Manning Publications.  
[5] Script del curso `Slides/pages/clase_4.py` — filtros Sobel/Laplaciano, `Conv2D + BatchNorm + MaxPooling` y visualización de mapas de activación.  
[6] Slidev — componente interactivo `ConvolutionSimulator.vue` (deck de la Sesión 4).